# Predicted-drift A-LQR ablation

Compare no drift cancellation, measured cancellation from an independent unsteered model, and predicted cancellation without that extra model. Keeping the gain operator fixed while changing the predictor separates the effect of the LQR gains from the accuracy of drift prediction. Measured and no-cancellation references come from notebook 03; predicted runs use its concept direction, cached Mean Jacobian, random-operator construction, costs, and sampling settings.

At steered blocks:
$$u_l = K_l\left(\alpha_l v_l - w(I-v_lv_l^\top)\hat d_l\right),$$
$$\hat d_0=0,\qquad \hat d_{l+1}=P_l\hat d_l+u_l.$$

Notation is for one token at the last position in one forward pass; vectors are in $\mathbb{R}^H$.

| Symbol | Meaning |
|---|---|
| $\mathbb{R}$ | Real numbers. |
| $l, L, H$ | Block index $l=0,\ldots,L-1$, number of blocks, and hidden size. |
| $\|\cdot\|$, $\top$ | Euclidean norm and transpose. |
| $x_l$ | Hidden state entering block $l$ in the steered run. |
| $x_l^{\mathrm{nom}}$ | Hidden state entering block $l$ in an unsteered run on the same text. |
| $d_l=x_l-x_l^{\mathrm{nom}}$ | True deviation caused by steering. |
| $\hat d_l$ | The controller's prediction of $d_l$; no unsteered run is needed. |
| $E_l$ | Concept vector: mean on concept prompts minus mean on other prompts. |
| $v_l=E_l/\|E_l\|$ | Unit concept direction. |
| $I, y$ | Identity matrix and an arbitrary hidden-state vector; $(I-v_lv_l^\top)y$ is the part of $y$ perpendicular to $v_l$. |
| $\lambda, \beta_l^*=\lambda\|E_l\|$ | Steering strength and target concept coordinate. |
| $\alpha_l=\beta_l^*-v_l^\top x_l$ | Concept error. |
| $u_l$ | Push added to block $l$'s output; zero at unsteered blocks. |
| $A_l$ | Gain operator: the linear block model used by Riccati to compute the gain. |
| $K_l$ | LQR gain at block $l$. |
| $Q, R, Q_F$ | State, control, and terminal costs used by Riccati; the configured scalars weight identity matrices. |
| $P_l$ | Predictor used to update $\hat d_l$; it may differ from $A_l$. |
| $w$ | Drift-cancellation weight. |
| $m$ (`every`) | Steer every $m$-th block within the selected layer range. |

$w=0$ reproduces plain A-LQR exactly. “Bookkeeping” means $P_l=I$: the predicted deviation is the sum of past pushes within this forward pass. It resets at block 0 on every forward pass; it does not carry deviations through the decode cache.

## 1. Config

In [ ]:
from pathlib import Path
import gc, importlib, random, sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'llm' / 'steering.py').is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import llm.model, llm.steering, llm.predicted_drift, llm.trace, llm.judge
for module in (llm.model, llm.steering, llm.predicted_drift, llm.trace, llm.judge):
    importlib.reload(module)
from llm.model import load_model, blocks
from llm.steering import LQRSteering, sample
from llm.predicted_drift import PredictedDriftLQRSteering
from llm.trace import trace, completion_metrics
from llm.judge import build_judge, concept_criterion, COHERENCE, repetition

MODEL_NAME = 'Qwen/Qwen3-4B'
CONCEPT = 'football'
PROMPT = 'Once upon a time'
SEED = 0
DIRECTION_SAMPLES = 200
JACOBIAN_SAMPLES = 50
MAX_CTX = 32
BATCH_SIZE = 8
RANDOM_SEED = 0
Q, R, QF = 0.1, 10, 0.1
STEER_LAST_N_LAYERS = None
MAX_NEW_TOKENS = 96
TEMPERATURE, TOP_P = 0.7, 0.95
JUDGE_BATCH = 512

EVERY = (1, 2)
CONDITIONS = [('Identity', 'local_jacobian'), ('Mean Jacobian', 'local_jacobian')]
DRIFT_WEIGHTS = (2.0,)
PILOT = True
TRIALS = 20 if PILOT else 100
LAMBDAS = (3, 4, 5, 6) if PILOT else (1, 2, 3, 4, 5, 6, 7, 8)
GRADE = False
MEASURED_CSV = (REPO_ROOT / 'artifacts' / MODEL_NAME.rsplit('/', 1)[-1].lower() / 'drift_lqr'
                / 'full' / f'q{Q:g}_r{R:g}_qf{QF:g}' / f'drift_ablation_{CONCEPT}.csv')


## 2. Load model and paths

For saved-CSV judging or analysis, run Config and the paths cell below, then skip model loading, operators, validation, and the sweep. Start at Judge to rescore, or References to analyze existing judgments.

Re-running the sweep overwrites results in the same pilot/full and cost directory.

In [ ]:
ARTIFACTS = REPO_ROOT / 'artifacts' / MODEL_NAME.rsplit('/', 1)[-1].lower()
RUN_DIR = ARTIFACTS / 'predicted_drift' / ('local_jacobian_pilot' if PILOT else 'local_jacobian_full') / f'q{Q:g}_r{R:g}_qf{QF:g}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_KWARGS = dict(max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P)
print('Results:', RUN_DIR)


In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
tokenizer, model = load_model(MODEL_NAME)
NUM_LAYERS = len(blocks(model))
HIDDEN = model.get_input_embeddings().weight.shape[1]
PROMPT_LEN = len(tokenizer(PROMPT)['input_ids'])
print(f'{MODEL_NAME}: {NUM_LAYERS} blocks, hidden {HIDDEN}, {next(model.parameters()).dtype}')


## 3. Concept direction and operators

Read the same caches as notebook 03; do not recompute or overwrite them. Random dynamics use its identity-plus-scaled-Gaussian construction.

In [ ]:
E_PATH = ARTIFACTS / f'E_{CONCEPT}_n{DIRECTION_SAMPLES}_seed{SEED}.pt'
A_PATH = ARTIFACTS / f'A_{CONCEPT}_n{JACOBIAN_SAMPLES}_ctx{MAX_CTX}_seed{SEED}.pt'
E = torch.load(E_PATH, map_location='cpu', weights_only=True)
A_jac = torch.load(A_PATH, map_location='cpu', weights_only=True)

B_random = torch.randn(NUM_LAYERS, HIDDEN, HIDDEN, generator=torch.Generator().manual_seed(RANDOM_SEED))
size = torch.linalg.matrix_norm(A_jac - torch.eye(HIDDEN)) / torch.linalg.matrix_norm(B_random)
A_random = torch.eye(HIDDEN) + size[:, None, None] * B_random
del B_random

OPERATORS = {'Identity': torch.eye(HIDDEN).expand(NUM_LAYERS, -1, -1),
             'Mean Jacobian': A_jac, 'Random': A_random,
             'local_jacobian': 'local_jacobian', 'local_jacobian_nom': 'local_jacobian_nom'}
print('E:', E_PATH.name, tuple(E.shape))
print('Mean Jacobian:', A_PATH.name)


## 4. Validation

Zero drift weight must reproduce the original controller's token IDs with Mean Jacobian gains and predictor.

In [ ]:
base = LQRSteering(model, A_jac, E, Q, R, QF, steer_last_n_layers=STEER_LAST_N_LAYERS, every=1)
controller = PredictedDriftLQRSteering(
    model, A_jac, E, Q, R, QF, predictor=A_jac, drift_weight=0,
    steer_last_n_layers=STEER_LAST_N_LAYERS, every=1)
torch.manual_seed(SEED)
base_ids = sample(model, tokenizer, PROMPT, 1, controller=base, lmbda=3,
                  **{**SAMPLE_KWARGS, 'max_new_tokens': 8})
torch.manual_seed(SEED)
predicted_ids = sample(model, tokenizer, PROMPT, 1, controller=controller, lmbda=3,
                       **{**SAMPLE_KWARGS, 'max_new_tokens': 8})
assert torch.equal(base_ids, predicted_ids)
print('Zero-weight token equality passed.')
del base, controller, base_ids, predicted_ids
gc.collect()
torch.cuda.empty_cache()


## 5. Sweep

Sample the unsteered baseline first, then reuse one controller per gain/predictor/every combination across weights and strengths. Seeds follow the index in `LAMBDAS`, as in notebook 03. Replay metrics are consumed before reusing the controller; unsteered replay metrics remain missing. Replay in BF16 is an approximation to cached generation.

In [ ]:
sweep_batches = 1 + len(EVERY) * len(CONDITIONS) * len(DRIFT_WEIGHTS) * len(LAMBDAS)
with tqdm(total=sweep_batches, desc='Predicted-drift A-LQR sweep', unit='batch') as progress:
    torch.manual_seed(SEED)
    ids = sample(model, tokenizer, PROMPT, TRIALS, **SAMPLE_KWARGS)
    results = [{'every': 0, 'gain_operator': 'Unsteered', 'predictor': 'Unsteered',
                'drift_weight': 0.0, 'lambda': 0.0, 'completion': text.strip()}
               for text in tokenizer.batch_decode(ids[:, PROMPT_LEN:], skip_special_tokens=True)]
    del ids
    progress.update(1)
    for every in EVERY:
        for gain, predictor in CONDITIONS:
            controller = PredictedDriftLQRSteering(
                model, OPERATORS[gain], E, Q, R, QF, predictor=OPERATORS[predictor],
                steer_last_n_layers=STEER_LAST_N_LAYERS, every=every)
            for weight in DRIFT_WEIGHTS:
                controller.drift_weight = weight
                for i, lmbda in enumerate(LAMBDAS):
                    torch.manual_seed(SEED + i)
                    ids = sample(model, tokenizer, PROMPT, TRIALS,
                                 controller=controller, lmbda=lmbda, **SAMPLE_KWARGS)
                    tr = trace(model, controller, lmbda, ids, PROMPT_LEN)
                    batch_metrics = completion_metrics(tr)
                    del tr
                    texts = tokenizer.batch_decode(ids[:, PROMPT_LEN:], skip_special_tokens=True)
                    for text, metrics in zip(texts, batch_metrics):
                        results.append({'every': every, 'gain_operator': gain, 'predictor': predictor,
                                        'drift_weight': weight, 'lambda': lmbda, **metrics,
                                        'completion': text.strip()})
                    progress.set_postfix(every=every, gain=gain, predictor=predictor, weight=weight, lmbda=lmbda)
                    progress.update(1)
                    del ids, batch_metrics, texts
            del controller
            gc.collect()
            torch.cuda.empty_cache()

results = pd.DataFrame(results)
results.to_csv(RUN_DIR / f'predicted_drift_{CONCEPT}_unjudged.csv', index=False)


## 6. Judge

Free generation models before loading Granite. This cell reloads saved completions and can be rerun without generation; both criteria use deduplicated, length-sorted batches with progress bars.

In [ ]:
for name in ('model', 'nominal', 'controller', 'judge'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
judge = build_judge(batch_size=JUDGE_BATCH)
results = pd.read_csv(RUN_DIR / f'predicted_drift_{CONCEPT}_unjudged.csv', keep_default_na=False)
texts = results['completion'].tolist()
print(f'Judging {len(texts):,} completions ({len(set(texts)):,} unique), two criteria, batch size {JUDGE_BATCH}.')
results['concept'] = judge(concept_criterion(CONCEPT), PROMPT, texts, desc=f'Concept: {CONCEPT} (1/2)')
results['coherent'] = judge(COHERENCE, PROMPT, texts, desc='Coherence (2/2)')
results['both'] = results['concept'] & results['coherent']
results['repetition'] = results['completion'].map(repetition)
results.to_csv(RUN_DIR / f'predicted_drift_{CONCEPT}.csv', index=False)
del judge
gc.collect()
torch.cuda.empty_cache()


## 7. References

Load saved judgments, retaining measured-run weights zero and two and all their strengths. `none` is steered A-LQR without cancellation, not the unsteered baseline. The measured CSV is read-only. Its historical configuration is not stored in that CSV; configuration values here match the current notebook 03.

In [ ]:
gain_names = list(dict.fromkeys(gain for gain, predictor in CONDITIONS))
measured = pd.read_csv(MEASURED_CSV)
keep = measured['every'].eq(0) | (
    measured['every'].isin(EVERY) & measured['operator'].isin(gain_names)
    & measured['drift_weight'].isin((0, 2)))
measured = measured.loc[keep].rename(columns={'operator': 'gain_operator'}).copy()
measured['source'] = np.where(measured['drift_weight'].eq(0), 'none', 'measured')
measured['predictor'] = measured['source']
results = pd.read_csv(RUN_DIR / f'predicted_drift_{CONCEPT}.csv')
results['source'] = 'predicted'
all_results = pd.concat([measured, results], ignore_index=True)


## 8. Summary table

Means pool completions within each cell. Energy includes both its mean and median; rate error bars are binomial standard errors, not uncertainty estimates for paired differences.

In [ ]:
keys = ['source', 'every', 'gain_operator', 'predictor', 'drift_weight', 'lambda']
groups = all_results.groupby(keys, sort=False)
summary = groups[['concept', 'coherent', 'both', 'repetition', 'along_v', 'change', 'orthogonal', 'push_energy']].mean()
summary['push_energy_median'] = groups['push_energy'].median()
summary['n'] = groups.size()
summary['concept_se'] = np.sqrt(summary['concept'] * (1 - summary['concept']) / summary['n'])
summary['coherent_se'] = np.sqrt(summary['coherent'] * (1 - summary['coherent']) / summary['n'])
summary = summary.reset_index()
summary.to_csv(RUN_DIR / f'predicted_drift_{CONCEPT}_summary.csv', index=False)
print(summary.round(4).to_string(index=False))


## 9. Coherence versus concept rate

One panel per gain operator; curves distinguish reference sources, predictors, and weights. Labels give strength, with one standard error on coherence only. The black star is this run's separately sampled unsteered baseline.

In [ ]:
def curve_label(source, predictor, weight):
    name = f'predicted: {predictor}' if source == 'predicted' else source
    return f'{name}, w={weight:g}'

steered = summary.loc[summary['every'].ne(0)]
baseline = summary.loc[summary['every'].eq(0) & summary['source'].eq('predicted')].iloc[0]
color_keys = list(steered[['gain_operator', 'source']].drop_duplicates().itertuples(index=False, name=None))
cmap = plt.get_cmap('tab10', len(color_keys))
colors = {key: cmap(i) for i, key in enumerate(color_keys)}
line_patterns = ('-', '--', ':', '-.', (0, (3, 1, 1, 1)))
styles = {name: line_patterns[i % len(line_patterns)] for i, name in enumerate(steered['predictor'].unique())}
marker_shapes = ('o', 's', '^', 'D', 'v', 'P', 'X')
markers = {weight: marker_shapes[i % len(marker_shapes)]
           for i, weight in enumerate(sorted(steered['drift_weight'].unique()))}

for every in EVERY:
    fig, axes = plt.subplots(1, len(gain_names), figsize=(5.5 * len(gain_names), 4.5),
                             sharex=True, sharey=True, squeeze=False, layout='constrained')
    for ax, gain in zip(axes[0], gain_names):
        panel = steered.loc[steered['every'].eq(every) & steered['gain_operator'].eq(gain)]
        for (source, predictor, weight), curve in panel.groupby(['source', 'predictor', 'drift_weight'], sort=False):
            curve = curve.sort_values('lambda')
            ax.errorbar(curve['concept'], curve['coherent'], yerr=curve['coherent_se'], capsize=2,
                        color=colors[gain, source], linestyle=styles[predictor], marker=markers[weight],
                        label=curve_label(source, predictor, weight))
            for lmbda, point in zip(curve['lambda'], curve.itertuples()):
                ax.annotate(f'{lmbda:g}', (point.concept, point.coherent), fontsize=7,
                            xytext=(3, 3), textcoords='offset points')
        ax.plot(baseline['concept'], baseline['coherent'], 'k*', ms=10, label='Unsteered', zorder=10)
        ax.set(title=f'Gain: {gain}', xlabel='Concept rate', xlim=(-0.05, 1.05), ylim=(-0.05, 1.05))
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7)
    axes[0, 0].set_ylabel('Coherence rate')
    fig.suptitle(f'{MODEL_NAME}, {CONCEPT}, every={every}; labels = lambda')
    fig.savefig(RUN_DIR / f'predicted_drift_{CONCEPT}_every{every}.png', dpi=150)
    plt.show()


## 10. Matched-concept table

Interpolate coherence and cell-median energy at the first crossing of each target while traversing increasing strength. Never sort by concept rate or extrapolate; an exact endpoint counts. Unreached targets and a zero measured-minus-none coherence gap give NaN. Gap recovered uses the same gain operator and actuation interval for both references and is not clipped.

In [ ]:
def at_concept(curve, target):
    points = curve.sort_values('lambda')[['concept', 'coherent', 'push_energy_median']].to_numpy(dtype=float)
    for i, left in enumerate(points):
        if left[0] == target:
            return left[1:].copy()
        if i + 1 < len(points):
            right = points[i + 1]
            if (left[0] - target) * (right[0] - target) < 0:
                fraction = (target - left[0]) / (right[0] - left[0])
                return left[1:] + fraction * (right[1:] - left[1:])
    return np.full(2, np.nan)

matched_rows = []
for (every, gain, predictor, weight), curve in steered.loc[steered['source'].eq('predicted')].groupby(
        ['every', 'gain_operator', 'predictor', 'drift_weight'], sort=False):
    references = steered.loc[steered['every'].eq(every) & steered['gain_operator'].eq(gain)]
    for target in (0.75, 0.9):
        coherent, energy = at_concept(curve, target)
        none_coherent, none_energy = at_concept(references.loc[references['source'].eq('none')], target)
        measured_coherent, measured_energy = at_concept(references.loc[references['source'].eq('measured')], target)
        gap = measured_coherent - none_coherent
        matched_rows.append({'every': every, 'gain_operator': gain, 'predictor': predictor,
                             'drift_weight': weight, 'target': target, 'coherent': coherent,
                             'push_energy_median': energy, 'none_coherent': none_coherent,
                             'none_push_energy_median': none_energy, 'measured_coherent': measured_coherent,
                             'measured_push_energy_median': measured_energy,
                             'gap_recovered': (coherent - none_coherent) / gap if np.isfinite(gap) and gap != 0 else np.nan})
matched = pd.DataFrame(matched_rows)
matched.to_csv(RUN_DIR / f'predicted_drift_{CONCEPT}_matched.csv', index=False)
print(matched.round(4).to_string(index=False))


## 11. Diagnostic figures

As in notebook 03, plot outcome rates against final along-concept shift, orthogonal change, normalized push energy, and total change, then plot orthogonal change against along-concept shift. Color identifies gain/source; line style identifies predictor and marker identifies weight. Metrics average valid generated-token positions; energy is the sum of squared pushes divided by squared block-input norms.

In [ ]:
METRICS = {'along_v': 'Along-v shift (final layer)', 'orthogonal': 'Off-v change (relative)',
           'push_energy': 'Push energy', 'change': 'Total change (relative)'}
OUTCOMES = {'coherent': 'Coherence rate', 'concept': 'Concept rate'}
for every in EVERY:
    block = steered.loc[steered['every'].eq(every)]
    fig, axes = plt.subplots(len(OUTCOMES), len(METRICS), figsize=(4.2 * len(METRICS), 3.6 * len(OUTCOMES)),
                             sharey='row', squeeze=False)
    for (gain, source, predictor, weight), curve in block.groupby(
            ['gain_operator', 'source', 'predictor', 'drift_weight'], sort=False):
        curve = curve.sort_values('lambda')
        for row, outcome in zip(axes, OUTCOMES):
            for ax, metric in zip(row, METRICS):
                ax.errorbar(curve[metric], curve[outcome], yerr=curve[f'{outcome}_se'],
                            color=colors[gain, source], linestyle=styles[predictor], marker=markers[weight],
                            ms=3, capsize=2, label=f'{gain}, {curve_label(source, predictor, weight)}')
    for row, ylabel in zip(axes, OUTCOMES.values()):
        row[0].set_ylabel(ylabel)
        row[list(METRICS).index('push_energy')].set_xscale('log')
        for ax, xlabel in zip(row, METRICS.values()):
            ax.set_xlabel(xlabel)
            ax.grid(alpha=0.2)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=3, fontsize=7)
    fig.suptitle(f'{MODEL_NAME}, {CONCEPT}, every={every}')
    fig.tight_layout()
    fig.savefig(RUN_DIR / f'predicted_drift_{CONCEPT}_metrics_every{every}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(EVERY), figsize=(4.5 * len(EVERY), 4), sharex=True, sharey=True, squeeze=False)
for ax, every in zip(axes[0], EVERY):
    for (gain, source, predictor, weight), curve in steered.loc[steered['every'].eq(every)].groupby(
            ['gain_operator', 'source', 'predictor', 'drift_weight'], sort=False):
        curve = curve.sort_values('lambda')
        ax.plot(curve['along_v'], curve['orthogonal'], color=colors[gain, source],
                linestyle=styles[predictor], marker=markers[weight], ms=3,
                label=f'{gain}, {curve_label(source, predictor, weight)}')
    ax.set(title=f'every={every}', xlabel='Along-v shift (final layer)')
    ax.grid(alpha=0.2)
axes[0, 0].set_ylabel('Off-v change (relative)')
handles, labels = axes[0, -1].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=3, fontsize=7)
fig.tight_layout()
fig.savefig(RUN_DIR / f'predicted_drift_{CONCEPT}_aim.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Optional grading

Only `GRADE=True` loads an independent nominal model, using the same configuration and tokens. Repeat the configured conditions, strengths, weights, and actuation intervals without replay; control still uses predicted drift. For each steered block and generation step, record batch means and valid counts for
$$\frac{\|(I-v_lv_l^\top)d_l\|}{\|x_l\|},\qquad
\frac{\|(I-v_lv_l^\top)\hat d_l\|}{\|x_l\|},\qquad
\frac{\|(I-v_lv_l^\top)(\hat d_l-d_l)\|}{\|(I-v_lv_l^\top)d_l\|}.$$
The CSV retains generation steps; printed and plotted summaries pool valid observations into step 0 versus later steps. Ignore padding after the first EOS, but include the step emitting it. Zero measured drift makes relative error undefined, not zero. The figure separates actuation intervals, strengths, and weights into panels. Skipping this section loads no nominal model and changes no earlier output.

In [ ]:
if GRADE:
    for name in ('model', 'nominal', 'controller', 'judge'):
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, grading_model = load_model(MODEL_NAME)
    nominal = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=next(grading_model.parameters()).dtype,
        device_map=str(grading_model.device),
        attn_implementation=grading_model.config._attn_implementation,
    ).eval()
    prompt_len = len(tokenizer(PROMPT)['input_ids'])
    eos = grading_model.generation_config.eos_token_id
    eos = [token for token in (eos if isinstance(eos, list) else [eos]) if token is not None]
    fields = ('measured_relative_drift', 'predicted_relative_drift', 'relative_error', 'cosine_perp')
    grading_rows = []
    total = len(EVERY) * len(CONDITIONS) * len(DRIFT_WEIGHTS) * len(LAMBDAS)
    try:
        with tqdm(total=total, desc='Prediction grading', unit='batch') as progress:
            for every in EVERY:
                for gain, predictor in CONDITIONS:
                    controller = PredictedDriftLQRSteering(
                        grading_model, OPERATORS[gain], E, Q, R, QF, predictor=OPERATORS[predictor],
                        nominal=nominal, steer_last_n_layers=STEER_LAST_N_LAYERS, every=every)
                    for weight in DRIFT_WEIGHTS:
                        controller.drift_weight = weight
                        for i, lmbda in enumerate(LAMBDAS):
                            torch.manual_seed(SEED + i)
                            ids = sample(grading_model, tokenizer, PROMPT, TRIALS,
                                         controller=controller, lmbda=lmbda, **SAMPLE_KWARGS)
                            generated = ids[:, prompt_len:].cpu()
                            is_eos = torch.isin(generated, torch.tensor(eos, dtype=generated.dtype)).long()
                            valid = ((is_eos.cumsum(dim=1) - is_eos) == 0).numpy()
                            log = controller.prediction_log
                            values = torch.stack([torch.stack([entry[field] for field in fields], dim=-1)
                                                  for entry in log]).float().cpu().numpy()[:, :, 0]
                            for entry, batch in zip(log, values):
                                step, layer = entry['step'], entry['layer']
                                batch = batch[valid[:, step]]
                                counts = np.isfinite(batch).sum(axis=0)
                                means = np.divide(np.nansum(batch, axis=0), counts,
                                                  out=np.full(len(fields), np.nan), where=counts > 0)
                                row = {'every': every, 'gain_operator': gain, 'predictor': predictor,
                                       'drift_weight': weight, 'lambda': lmbda, 'layer': layer, 'step': step}
                                for field, mean, count in zip(fields, means, counts):
                                    row[field] = mean
                                    row[f'{field}_n'] = int(count)
                                grading_rows.append(row)
                            progress.set_postfix(every=every, gain=gain, predictor=predictor, weight=weight, lmbda=lmbda)
                            progress.update(1)
                            del ids, log, values, entry
                    del controller
                    gc.collect()
                    torch.cuda.empty_cache()
    finally:
        for name in ('grading_model', 'nominal', 'controller', 'ids'):
            globals().pop(name, None)
        gc.collect()
        torch.cuda.empty_cache()

    grading = pd.DataFrame(grading_rows)
    grading.to_csv(RUN_DIR / f'predicted_drift_{CONCEPT}_grading.csv', index=False)
    phases = grading.assign(phase=np.where(grading['step'].eq(0), 'step 0', 'steps >=1'))
    phase_keys = ['every', 'gain_operator', 'predictor', 'drift_weight', 'lambda', 'layer', 'phase']
    for field in fields:
        phases[field] *= phases[f'{field}_n']
    grading_summary = phases.groupby(phase_keys, sort=False)[
        [*fields, *(f'{field}_n' for field in fields)]].sum(min_count=1)
    for field in fields:
        grading_summary[field] /= grading_summary[f'{field}_n'].replace(0, np.nan)
    grading_summary = grading_summary.reset_index()
    last = grading_summary['layer'].eq(grading_summary.groupby(phase_keys[:5])['layer'].transform('max'))
    print('Last steered block (zero-based):')
    print(grading_summary.loc[last, [*phase_keys, *fields]].round(4).to_string(index=False))

    panels = [(weight, lmbda) for weight in DRIFT_WEIGHTS for lmbda in LAMBDAS]
    fig, axes = plt.subplots(len(EVERY), len(panels), figsize=(4.5 * len(panels), 3.5 * len(EVERY)),
                             sharex=True, sharey=True, squeeze=False)
    grade_cmap = plt.get_cmap('tab10', len(CONDITIONS))
    for row, every in zip(axes, EVERY):
        for ax, (weight, lmbda) in zip(row, panels):
            panel = grading_summary.loc[grading_summary['every'].eq(every)
                                        & grading_summary['drift_weight'].eq(weight)
                                        & grading_summary['lambda'].eq(lmbda)]
            for j, (gain, predictor) in enumerate(CONDITIONS):
                for phase, style in (('step 0', '-'), ('steps >=1', '--')):
                    curve = panel.loc[panel['gain_operator'].eq(gain) & panel['predictor'].eq(predictor)
                                      & panel['phase'].eq(phase)].sort_values('layer')
                    ax.plot(curve['layer'], curve['relative_error'], color=grade_cmap(j), linestyle=style,
                            label=f'{gain} / {predictor}, {phase}')
            ax.set(title=f'every={every}, w={weight:g}, lambda={lmbda:g}', xlabel='Block index (zero-based)')
            ax.grid(alpha=0.2)
        row[0].set_ylabel('Relative prediction error')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=3, fontsize=7)
    fig.tight_layout()
    fig.savefig(RUN_DIR / f'predicted_drift_{CONCEPT}_grading.png', dpi=150, bbox_inches='tight')
    plt.show()
